# Cision One API Explorer
**Organisation:** GoDaddy.com, LLC

Interactive notebook for pulling and reviewing data from Cision One mention streams.

**API v2 Endpoints used:**
| Endpoint | Description |
|---|---|
| `GET /public/api/v2/streams` | List all mention streams |
| `GET /public/api/v2/mentions/{streamId}` | Fetch mentions for a stream |
| `GET /public/api/v2/streams/{streamId}/stats` | Aggregated stats for a stream |

**Docs:** https://developers.cision.one/docs/api/v2

## 1. Setup

In [ ]:
import requests
import json
import time
import csv
from datetime import datetime, timedelta, timezone
from collections import Counter
from typing import Optional

import pandas as pd

BASE_URL = "https://api.cision.one"

# ⬇️ Paste your API token here
API_TOKEN = "YOUR_API_TOKEN_HERE"

session = requests.Session()
session.headers.update({
    "X-Auth-Token": API_TOKEN,
    "Accept": "application/json",
})

print("✓ Session configured")

## 2. Helper Functions

> **Rate limit note:** The Cision One API is limited to **~10 requests per minute**
> per API key / IP. All functions below include automatic throttling and retry-on-429
> handling so you can run them hands-off. Expect ~7 seconds per request; a 30-day
> pull on a busy stream may take several minutes.

In [ ]:
# ── Rate-limit-aware request wrapper ───────────────────────────────────

REQUEST_INTERVAL = 7.0  # seconds between requests (≈ 8–9 req/min, safe under the 10/min cap)
MAX_RETRIES = 5         # max retries on a 429 before giving up
_last_request_time = 0.0  # module-level timestamp of most recent API call


def _throttled_get(url: str, params: dict) -> requests.Response:
    """
    GET with pre-request throttle + automatic 429 retry with exponential backoff.
    Respects the Retry-After header when present.
    """
    global _last_request_time

    for attempt in range(1, MAX_RETRIES + 1):
        # Pre-request throttle: wait until REQUEST_INTERVAL has elapsed
        elapsed = time.time() - _last_request_time
        if elapsed < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - elapsed)

        _last_request_time = time.time()
        resp = session.get(url, params=params)

        if resp.status_code != 429:
            return resp

        # 429 → back off
        retry_after = resp.headers.get("Retry-After")
        if retry_after:
            wait = float(retry_after)
        else:
            wait = REQUEST_INTERVAL * (2 ** (attempt - 1))  # exponential backoff
        print(f"  ⏳ Rate-limited (429). Retry {attempt}/{MAX_RETRIES} — waiting {wait:.0f}s …")
        time.sleep(wait)

    # All retries exhausted — raise the last 429
    resp.raise_for_status()


# ── Core API functions ────────────────────────────────────────────────

def list_streams() -> list[dict]:
    """Return all Mention Streams for your organisation."""
    resp = _throttled_get(f"{BASE_URL}/public/api/v2/streams", {"format": "json"})
    resp.raise_for_status()
    return resp.json()


def get_mentions(
    stream_id: int,
    after: str,
    before: str,
    page: int = 1,
    page_size: int = 100,
    sort_field: str = "timestamp",
    sort_order: str = "desc",
) -> list[dict]:
    """Fetch a single page of mentions from a stream."""
    resp = _throttled_get(
        f"{BASE_URL}/public/api/v2/mentions/{stream_id}",
        {
            "filter[range][after]": after,
            "filter[range][before]": before,
            "pagination[page]": page,
            "pagination[page_size]": page_size,
            "sort[field]": sort_field,
            "sort[order]": sort_order,
            "format": "json",
        },
    )
    resp.raise_for_status()
    return resp.json()


def get_all_mentions(
    stream_id: int,
    after: str,
    before: str,
    page_size: int = 500,
    max_pages: int = 9,
) -> list[dict]:
    """
    Auto-paginate through mentions for a SINGLE date window.
    Caps at page_size * max_pages to stay within the API's 5,000-mention ceiling.
    Returns whatever it collected if a 400 (ceiling hit) occurs.
    """
    all_mentions = []
    page = 1
    while page <= max_pages and (page * page_size) <= 5000:
        try:
            batch = get_mentions(stream_id, after, before, page=page, page_size=page_size)
        except requests.HTTPError as e:
            if e.response.status_code == 400:
                print(f"  ⚠️  Hit API ceiling at page {page} — returning {len(all_mentions)} mentions for this window")
                break
            raise
        if not batch:
            break
        all_mentions.extend(batch)
        print(f"  Page {page}: {len(batch)} mentions (running total: {len(all_mentions)})")
        if len(batch) < page_size:
            break
        page += 1
    return all_mentions


def get_all_mentions_chunked(
    stream_id: int,
    after: str,
    before: str,
    chunk_days: int = 1,
    page_size: int = 500,
) -> list[dict]:
    """
    Bypass the 5,000-mention API ceiling by splitting the date range into
    smaller time windows (default: 1 day each) and paginating within each.
    All rate-limit handling happens inside _throttled_get — this just orchestrates.
    """
    from dateutil import parser as dtparser

    start = dtparser.isoparse(after)
    end   = dtparser.isoparse(before)
    chunk = timedelta(days=chunk_days)

    all_mentions = []
    window_start = start
    window_num = 0
    total_windows = int((end - start) / chunk) + 1

    print(f"📊 Fetching ~{total_windows} daily windows at ~{REQUEST_INTERVAL}s/request …")
    print(f"   (Rate limit: ~10 requests/min — this will take a few minutes)\n")

    while window_start < end:
        window_end = min(window_start + chunk, end)
        window_num += 1
        w_after  = window_start.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        w_before = window_end.strftime("%Y-%m-%dT%H:%M:%S.000Z")

        print(f"📅 Window {window_num}/{total_windows}: {w_after[:10]} → {w_before[:10]}")
        batch = get_all_mentions(
            stream_id, w_after, w_before, page_size=page_size,
        )
        all_mentions.extend(batch)
        print(f"  Window total: {len(batch)} | Running total: {len(all_mentions)}")
        window_start = window_end

    # Deduplicate by mention ID (in case of boundary overlap)
    seen = set()
    deduped = []
    for m in all_mentions:
        mid = m.get("id")
        if mid not in seen:
            seen.add(mid)
            deduped.append(m)
    if len(deduped) < len(all_mentions):
        print(f"\nRemoved {len(all_mentions) - len(deduped)} duplicate mentions across windows")

    print(f"\n✅ Total unique mentions: {len(deduped)}")
    return deduped


def get_stream_stats(stream_id: int, after: str, before: str) -> dict:
    """Return aggregated stats for a stream within a date range."""
    resp = _throttled_get(
        f"{BASE_URL}/public/api/v2/streams/{stream_id}/stats",
        {
            "filter[range][after]": after,
            "filter[range][before]": before,
            "format": "json",
        },
    )
    resp.raise_for_status()
    return resp.json()


def mentions_to_df(mentions: list[dict]) -> pd.DataFrame:
    """Flatten mentions into a clean DataFrame.
    Handles both online/print mentions (which use 'excerpt') and
    broadcast TV/radio mentions (which use 'transcript' + broadcast-specific fields).
    A unified 'content' column merges excerpt and transcript for easy searching.
    """
    rows = []
    for m in mentions:
        impact = m.get("impactScore")
        # impactScore can be a list of dicts, a single dict, or None/empty
        if isinstance(impact, list) and len(impact) > 0:
            impact_obj = impact[0]
        elif isinstance(impact, dict):
            impact_obj = impact
        else:
            impact_obj = {}
        row = {
            "id": m.get("id"),
            "type": m.get("type"),
            "publishedAt": m.get("publishedAt"),
            "medium": m.get("medium"),
            "title": m.get("title"),
            "author": m.get("author"),
            "source": m.get("source"),
            "url": m.get("url"),
            "internalLink": m.get("internalLink"),
            "sentiment": m.get("sentiment"),
            "audience": m.get("audience"),
            "advertisingValue": m.get("advertisingValue"),
            "wordCount": m.get("wordCount"),
            "domainAuthority": m.get("domainAuthority"),
            "country": m.get("locationCountry"),
            "state": m.get("locationState"),
            "city": m.get("locationCity"),
            "language": m.get("languageCode"),
            "keywords": "; ".join(m.get("keywords", []) or []),
            # Online / print content
            "excerpt": m.get("excerpt"),
            # Broadcast (TV / radio) content
            "transcript": m.get("transcript"),
            "archivedLink": m.get("archivedLink"),
            "localViewershipAudience": m.get("localViewershipAudience"),
            "nationalViewershipAudience": m.get("nationalViewershipAudience"),
            "localViewershipAdValue": m.get("localViewershipAdValue"),
            "nationalViewershipAdValue": m.get("nationalViewershipAdValue"),
            # Impact
            "impactScore": impact_obj.get("score"),
            "impactGrade": impact_obj.get("grade"),
        }
        # Unified content column: excerpt for online/print, transcript for TV/radio
        row["content"] = row["excerpt"] or row["transcript"]
        rows.append(row)
    df = pd.DataFrame(rows)
    if "publishedAt" in df.columns:
        df["publishedAt"] = pd.to_datetime(df["publishedAt"], errors="coerce")
    return df

print("✓ Helper functions loaded")

## 3. List All Mention Streams

This maps the stream names you see in the Cision One sidebar (PRODUCTS T1 → AGI, ANS, Airo, Commerce, Other; CORPORATE T1 → Brand, Finance, Thought Leadership; etc.) to their numeric IDs needed for the API.

In [ ]:
streams = list_streams()
streams_df = pd.DataFrame(streams)

# Reorder columns for readability
col_order = ["id", "label", "archived", "queryStyle", "keywords", "excludedKeywords",
             "onlineContent", "printContent", "tvContent", "radioContent",
             "socialContent", "podcastContent", "magazineContent"]
col_order = [c for c in col_order if c in streams_df.columns]
streams_df = streams_df[col_order]

print(f"Found {len(streams_df)} streams\n")
streams_df

## 4. Select a Stream & Date Range

Update `STREAM_ID` with one of the IDs from the table above. Adjust the date range as needed.

In [ ]:
# ⬇️ Replace with a stream ID from the table above
STREAM_ID = 0  # e.g. the "Brand" stream under CORPORATE T1

# Date range — default: last 30 days
now = datetime.now(timezone.utc)
AFTER  = (now - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S.000Z")
BEFORE = now.strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Stream ID: {STREAM_ID}")
print(f"Date range: {AFTER} → {BEFORE}")

## 5. Stream Statistics

Aggregated sentiment, audience, and advertising value breakdown.

In [ ]:
stats = get_stream_stats(STREAM_ID, AFTER, BEFORE)

print(f"Stream: {stats.get('streamLabel', '?')}  (ID {stats.get('streamId', '?')})")
print(f"Media types: {', '.join(stats.get('media', []))}")
print()

# Sentiment distribution
sent_data = stats.get("sentimentAggregation", [])
if sent_data:
    sent_df = pd.DataFrame([
        {"Sentiment": s["label"], "Mentions": s["values"]["doc_count"]}
        for s in sent_data
    ])
    display(sent_df)

# Audience by type
aud_data = stats.get("audiencesByType", [])
if aud_data:
    print("\nAudience by media type:")
    for a in aud_data:
        print(f"  {a['label']}: {a['value']:,}")

# Ad values
ad_data = stats.get("advertisingValues", [])
if ad_data:
    print("\nAdvertising values:")
    for av in ad_data:
        print(f"  {av['label']}: count={av['values']['count']}, total={av['values']['total']}")

## 6. Fetch Mentions

The API caps at 5,000 mentions per request window. To get everything, we split the date range into
**daily chunks**, paginate within each, and deduplicate at the end. Adjust `chunk_days` if a single
day still exceeds 5,000 mentions (unlikely for most streams).

In [ ]:
raw_mentions = get_all_mentions_chunked(STREAM_ID, AFTER, BEFORE, chunk_days=1)
df = mentions_to_df(raw_mentions)
print(f"\n✓ {len(df)} mentions loaded into DataFrame")
df.head(10)

## 7. Explore the Data

### 7a. Breakdown by Medium

In [ ]:
df["medium"].value_counts()

### 7b. Sentiment Distribution

In [ ]:
df["sentiment"].describe()

In [ ]:
# Histogram of sentiment scores
df["sentiment"].dropna().plot.hist(
    bins=30,
    title="Sentiment Distribution",
    xlabel="Sentiment Score",
    figsize=(8, 3),
    color="#4a90d9",
    edgecolor="white",
)

### 7c. Top Sources by Volume

In [ ]:
df["source"].value_counts().head(15)

### 7d. Mentions Over Time

In [ ]:
daily = df.set_index("publishedAt").resample("D").size()
daily.plot(
    title="Daily Mention Volume",
    figsize=(10, 3),
    color="#4a90d9",
)

### 7e. Top Keywords

In [ ]:
# Explode the semicolon-delimited keywords column
kw = df["keywords"].dropna().str.split("; ").explode().str.strip()
kw = kw[kw != ""]
kw.value_counts().head(20)

### 7f. Highest Impact Mentions

In [ ]:
df.nlargest(10, "impactScore")[["publishedAt", "source", "title", "impactScore", "impactGrade", "audience", "sentiment"]]

### 7g. Negative Coverage Deep-Dive

In [ ]:
negative = df[df["sentiment"] < -0.2].sort_values("audience", ascending=False)
print(f"{len(negative)} negative mentions found\n")
negative[["publishedAt", "source", "title", "sentiment", "audience"]].head(10)

## 8. Multi-Stream Comparison

Compare stats across multiple streams (e.g. all your PRODUCTS T1 sub-streams).

In [ ]:
# ⬇️ Replace with real stream IDs from Step 3
COMPARE_IDS = {
    # "AGI": 11111,
    # "ANS": 22222,
    # "Airo": 33333,
    # "Commerce": 44444,
}

comparison_rows = []
for label, sid in COMPARE_IDS.items():
    try:
        st = get_stream_stats(sid, AFTER, BEFORE)
        total_audience = sum(a["value"] for a in st.get("audiencesByType", []))
        sent_counts = {s["label"]: s["values"]["doc_count"] for s in st.get("sentimentAggregation", [])}
        total_mentions = sum(sent_counts.values())
        comparison_rows.append({
            "Stream": label,
            "Total Mentions": total_mentions,
            "Total Audience": total_audience,
            "Positive": sent_counts.get("Positive", 0),
            "Balanced": sent_counts.get("Balanced", 0),
            "Negative": sent_counts.get("Negative", 0),
        })
        time.sleep(0.3)
    except Exception as e:
        print(f"  ⚠️ {label} (ID {sid}): {e}")

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows).set_index("Stream")
    display(comp_df)
    comp_df[["Positive", "Balanced", "Negative"]].plot.bar(
        stacked=True,
        title="Sentiment Comparison Across Streams",
        figsize=(10, 4),
        color=["#4caf50", "#90a4ae", "#e53935"],
    )
else:
    print("Uncomment and fill in COMPARE_IDS above to use this section.")

## 9. Export to CSV

In [ ]:
output_path = "mentions_export.csv"
df.to_csv(output_path, index=False)
print(f"✓ Exported {len(df)} mentions to {output_path}")

## 10. Raw JSON Inspection

Peek at the raw API response for a single mention (useful for understanding all available fields).

In [ ]:
if raw_mentions:
    print(json.dumps(raw_mentions[0], indent=2, default=str))
else:
    print("No mentions to inspect — run Step 6 first.")